# 🔧 ASR Correction Model — Fine-Tuning Notebook

**Pipeline:**
1. Data Exploration — view existing training data & stats
2. Generate Hard Negatives — "don't correct" examples from common word + vocab matches  
3. Merge & Balance — combine all sources with proper ratio
4. Train LoRA — fine-tune Qwen2.5-7B with MLX
5. Evaluate — test on known cases
6. Deploy — copy adapters to production

In [1]:
# Setup — run once
import json, os, sys, time, logging
from pathlib import Path
from collections import Counter

# Project root
ROOT = Path("..").resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
print(f"Working dir: {os.getcwd()}")

Working dir: /Users/sanukathamuditha/Documents/ScreenApp/multimodal-correction-in-stt-systems


## 1. Data Exploration
See what training data we have and its balance.

In [2]:
# Count existing training data by source and type
data_files = {
    "LibriSpeech pairs": "data/training_pairs/librispeech_pairs.jsonl",
    "AMI pairs": "data/training_pairs/ami_pairs.jsonl",
    "ScreenApp corrections": "asr_correction/collected_data/corrections.jsonl",
}

for name, path in data_files.items():
    if not Path(path).exists():
        print(f"  {name}: NOT FOUND")
        continue
    positives = negatives = 0
    categories = Counter()
    with open(path) as f:
        for line in f:
            if not line.strip(): continue
            entry = json.loads(line)
            meta = entry.get("metadata", {})
            if meta.get("is_negative"):
                negatives += 1
            else:
                positives += 1
            categories[meta.get("category", "unknown")] += 1
    total = positives + negatives
    neg_pct = negatives / total * 100 if total else 0
    print(f"  {name}: {total} total ({positives} positive, {negatives} negative = {neg_pct:.0f}%)")
    print(f"    Categories: {dict(categories.most_common(5))}")

# Current merged train/valid
for split in ["train", "valid"]:
    path = f"data/collected_data/{split}.jsonl"
    if Path(path).exists():
        with open(path) as f:
            count = sum(1 for _ in f)
        print(f"\n  Current {split}.jsonl: {count} examples")

  LibriSpeech pairs: 34244 total (34242 positive, 2 negative = 0%)
    Categories: {'proper_noun': 33763, 'content_word': 481}
  AMI pairs: 7652 total (604 positive, 7048 negative = 92%)
    Categories: {'content_word': 6636, 'proper_noun': 1016}
  ScreenApp corrections: 951 total (951 positive, 0 negative = 0%)
    Categories: {'person_name': 239, 'tech_term': 185, 'compliance': 119, 'feature': 92, 'company_name': 75}

  Current train.jsonl: 37830 examples

  Current valid.jsonl: 6676 examples


## 2. Generate Hard Negatives

Create "don't correct this" examples from common words that match vocab `known_errors`.
These teach the model that "can" is NOT "Canva", "other" is NOT "Otter", etc.

In [3]:
# Load domain vocabulary
from asr_correction.vocabulary import load_domain_vocab

domain_vocab = load_domain_vocab("asr_correction/domain_vocab.json")
print(f"Domain vocab: {len(domain_vocab)} terms")

# Show terms with known_errors that are common English words
from wordfreq import zipf_frequency

print("\n--- Potentially problematic known_errors (common words → domain terms) ---")
problematic = []
for term_info in domain_vocab:
    term = term_info["term"]
    for err in term_info.get("known_errors", []):
        freq = zipf_frequency(err.lower().strip(".,!?"), "en")
        if freq > 3.5:  # Common word
            problematic.append((err, term, freq, term_info.get("category", "?")))

problematic.sort(key=lambda x: -x[2])
for err, term, freq, cat in problematic[:30]:
    print(f"  '{err}' (freq={freq:.1f}) → '{term}' ({cat})")

Domain vocab: 3 terms


ModuleNotFoundError: No module named 'wordfreq'

In [ ]:
# Generate hard negative training examples
# Strategy: For each vocab term, find common English words that sound/look similar
# and create "don't correct" examples using those words in real transcript context.

import random, json
from pathlib import Path
from collections import Counter
from wordfreq import zipf_frequency
from rapidfuzz import fuzz, process as rfprocess

sys.path.insert(0, ".")
from training.prepare_data import SYSTEM_PROMPT, build_user_prompt, build_assistant_response
from asr_correction.vocabulary import load_domain_vocab

random.seed(42)

# Load domain vocab (proper format)
raw_vocab = json.load(open("asr_correction/domain_vocab.json"))
domain_vocab = raw_vocab.get("terms", raw_vocab) if isinstance(raw_vocab, dict) else raw_vocab
print(f"Domain vocab: {len(domain_vocab)} terms")

# Load transcripts
transcripts = []
for path in ["data/transcripts/librispeech_all.jsonl", "data/transcripts/librispeech.jsonl"]:
    if Path(path).exists():
        with open(path) as f:
            for line in f:
                if line.strip():
                    text = json.loads(line).get("transcript_text", "")
                    if len(text) > 50:
                        transcripts.append(text)
        break
print(f"Loaded {len(transcripts)} transcripts")

# For each vocab term, find common words in transcripts that LOOK similar
# These are false positives the fuzzy matcher would catch
hard_negatives = []
context_window = 80

# Build list of all vocab terms for matching
all_terms = [(t["term"], t.get("category", "custom")) for t in domain_vocab]

# Common English words to check against vocab
# Extract unique words from transcripts
print("Extracting unique words from transcripts...")
word_set = set()
for text in transcripts[:500]:  # Sample for speed
    for w in text.split():
        clean = w.strip(".,!?;:'\"()[]").lower()
        if len(clean) >= 3:
            word_set.add(clean)

print(f"Found {len(word_set)} unique words")

# Find words that fuzzy-match vocab terms but are common English words
print("\nFinding false-positive matches...")
false_matches = []

for word in word_set:
    freq = zipf_frequency(word, "en")
    if freq < 3.0:  # Skip rare words (they might be real errors)
        continue
    
    # Check fuzzy similarity to each vocab term
    for term, category in all_terms:
        ratio = fuzz.ratio(word, term.lower())
        if ratio >= 60 and word != term.lower():
            false_matches.append({
                "word": word,
                "term": term,
                "category": category,
                "similarity": ratio,
                "word_freq": freq,
            })

# Sort by frequency (most common words first)
false_matches.sort(key=lambda x: (-x["word_freq"], -x["similarity"]))
print(f"Found {len(false_matches)} false-positive (common_word ≈ vocab_term) matches")
print("\nTop 20:")
for m in false_matches[:20]:
    print(f"  '{m['word']}' (freq={m['word_freq']:.1f}) ≈ '{m['term']}' ({m['category']}) sim={m['similarity']}")

# Generate training examples from these false matches
for match in false_matches:
    word = match["word"]
    term = match["term"]
    category = match["category"]
    count = 0
    
    for transcript in transcripts:
        t_lower = transcript.lower()
        idx = 0
        while count < 10:  # Max 10 per (word, term) pair
            pos = t_lower.find(word, idx)
            if pos == -1:
                break
            idx = pos + 1
            
            # Word boundary check
            if pos > 0 and t_lower[pos-1].isalnum():
                continue
            end_pos = pos + len(word)
            if end_pos < len(t_lower) and t_lower[end_pos].isalnum():
                continue
            
            # Extract context
            ctx_start = max(0, pos - context_window)
            ctx_end = min(len(transcript), pos + context_window)
            context = transcript[ctx_start:ctx_end]
            if len(context) < 20:
                continue
            
            entry = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(context, [term], category)},
                    {"role": "assistant", "content": build_assistant_response(
                        context, [], 0.99, False  # No changes, high confidence
                    )},
                ],
                "metadata": {
                    "source": "hard_negative",
                    "applied": False,
                    "term": term,
                    "category": category,
                    "error_found": word,
                    "is_negative": True,
                    "word_frequency": match["word_freq"],
                    "similarity": match["similarity"],
                },
            }
            hard_negatives.append(entry)
            count += 1
        
        if count >= 10:
            break

print(f"\n=== Generated {len(hard_negatives)} hard negative examples ===")
print(f"Top terms: {Counter(e['metadata']['term'] for e in hard_negatives).most_common(10)}")

In [ ]:
# Save hard negatives
output_path = "data/training_pairs/hard_negatives.jsonl"
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
    for entry in hard_negatives:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Saved {len(hard_negatives)} hard negatives to {output_path}")

## 3. Merge & Balance Data

Combine all sources: LibriSpeech pairs + AMI pairs + ScreenApp corrections + **hard negatives**.
Target: ~35-40% negative examples (up from 16%).

In [ ]:
!python training/merge_and_split.py \
  --sources data/training_pairs/librispeech_pairs.jsonl \
            data/training_pairs/ami_pairs.jsonl \
            data/training_pairs/hard_negatives.jsonl \
            asr_correction/collected_data/corrections.jsonl \
  --output-dir data/collected_data \
  --screenapp-upsample 3

In [ ]:
# Verify data balance
for split in ["train", "valid"]:
    path = f"data/collected_data/{split}.jsonl"
    pos = neg = 0
    with open(path) as f:
        for line in f:
            if not line.strip(): continue
            meta = json.loads(line).get("metadata", {})
            if meta.get("is_negative") or not meta.get("applied", True):
                neg += 1
            else:
                pos += 1
    total = pos + neg
    print(f"{split}: {total} total | {pos} positive ({pos/total*100:.0f}%) | {neg} negative ({neg/total*100:.0f}%)")

## 4. Train LoRA

Fine-tune Qwen2.5-7B-Instruct-4bit with the balanced dataset.
Adjust `ITERATIONS`, `LEARNING_RATE`, `BATCH_SIZE` below as needed.

In [ ]:
# ======= TRAINING CONFIG — EDIT THESE =======
ITERATIONS = 2000
LEARNING_RATE = 1e-5
BATCH_SIZE = 4
LORA_RANK = 16
LORA_LAYERS = 8
LORA_SCALE = 16.0
STEPS_PER_EVAL = 500
VAL_BATCHES = 25
# =============================================

from training.train_lora import TrainingConfig, train_with_mlx

config = TrainingConfig(
    iterations=ITERATIONS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    lora_rank=LORA_RANK,
    lora_layers=LORA_LAYERS,
    lora_scale=LORA_SCALE,
    steps_per_eval=STEPS_PER_EVAL,
    val_batches=VAL_BATCHES,
)

print(f"Config: iters={config.iterations}, lr={config.learning_rate}, bs={config.batch_size}")
print(f"LoRA: rank={config.lora_rank}, layers={config.lora_layers}, scale={config.lora_scale}")
print(f"Data: {config.data_dir}")
print(f"\nStarting training...")

result = train_with_mlx(config)

print(f"\n{'='*50}")
print(f"TRAINING COMPLETE")
print(f"Duration: {result['duration_seconds']/60:.1f} min")
print(f"Train examples: {result['data']['train_examples']}")
print(f"Valid examples: {result['data']['valid_examples']}")

## 5. Evaluate

Test the trained model on known cases — both real errors and false positives.

In [ ]:
# Load the freshly trained model
from asr_correction.model import load_model, unload_model, build_prompt, run_inference

# Force reload with new adapters
unload_model()
model, tokenizer = load_model(
    adapter_path="asr_correction/adapters",
    model_path="asr_correction/model_weights",
)
print(f"Model loaded: {model is not None}")

In [ ]:
# Test cases: (context, vocab_term, category, expected_action)
SYSTEM_PROMPT = "You are an ASR transcript correction model. Given a noisy ASR transcript segment and context signals, detect errors in context-critical terms and output the corrected transcript with changes noted."

test_cases = [
    # FALSE POSITIVES — model should say "no changes"
    ("we can give you the numbers", "Canva", "company_name", "NO CHANGE"),
    ("the other option is to wait", "Otter", "product_name", "NO CHANGE"),
    ("and then we proceed", "Andre", "person_name", "NO CHANGE"),
    ("but I think we should", "Bud", "person_name", "NO CHANGE"),
    ("so the thing is", "Bing", "product_name", "NO CHANGE"),
    ("it was a good result", "AWS", "tech_acronym", "NO CHANGE"),
    ("I can see the difference", "SEO", "tech_acronym", "NO CHANGE"),
    
    # REAL ERRORS — model should suggest corrections
    ("we use quadrant for vector search", "Qdrant", "tech_term", "CORRECT"),
    ("our rag pipeline processes documents", "RAG", "tech_term", "CORRECT"),
    ("the speed to text system", "speech", "tech_term", "CORRECT"),
]

print(f"{'Context':<45} {'Vocab':<10} {'Expected':<10} {'Model Says':<12} {'Conf':<6} {'OK?'}")
print("-" * 100)

correct = 0
for context, term, category, expected in test_cases:
    prompt = build_prompt(context, [term], category)
    result = run_inference(prompt, SYSTEM_PROMPT, model, tokenizer, max_tokens=256)
    
    changes = result.get("changes", [])
    confidence = result.get("confidence", 0.0)
    
    if changes:
        action = "CORRECT"
    else:
        action = "NO CHANGE"
    
    ok = action == expected
    correct += ok
    
    print(f"{context:<45} {term:<10} {expected:<10} {action:<12} {confidence:<6.2f} {'✓' if ok else '✗'}")

print(f"\n{correct}/{len(test_cases)} correct")

## 6. Deploy

Copy trained adapters to production path and update config.

In [ ]:
# Update adapter_config.json to match training config
import json

adapter_config = {
    "fine_tune_type": "lora",
    "model": "mlx-community/Qwen2.5-7B-Instruct-4bit",
    "num_layers": LORA_LAYERS,
    "lora_parameters": {
        "rank": LORA_RANK,
        "scale": LORA_SCALE,
        "dropout": 0.0,
        "keys": ["self_attn.q_proj", "self_attn.v_proj"],
    },
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "iters": ITERATIONS,
}

config_path = "asr_correction/adapters/adapter_config.json"
with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)

print(f"Updated {config_path}")
print(f"Adapters ready at: asr_correction/adapters/adapters.safetensors")
print(f"\nRestart the server with ./run.sh to use the new model")